In [ ]:
# import libraries
import requests 
import pandas as pd
import io 
import os 



In [27]:
# create a variable to store the file path
filepath_istat_data = '../raw/dati_istat.csv'

# create a logic to check if the file exists, if it does, read it into a dataframe, if not, fetch the data from the URL and save it to the file path
if os.path.exists(filepath_istat_data):
    df_istat_data = pd.read_csv(filepath_istat_data)
else:
    url='https://esploradati.istat.it/SDMXWS/rest/data/41_983'
    headers = {'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.content.decode('utf-8')
        df_istat_data = pd.read_csv(io.StringIO(data))
        df_istat_data.to_csv(filepath_istat_data, index=False)
    else:
        print(f"Failed to fetch data. Status code: {response.status_code}")


In [28]:
# display the first 5 rows of the dataframe
df_istat_data.head()

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# The SITUAS API requires a POST request with a JSON body and session cookie.
# Content-Type tells the server we are sending JSON data.
# The Cookie header authenticates the request using the browser session.
# If the cookie expires, a new one can be obtained by opening the SITUAS website,
# clicking "Esporta" and copying the Cookie value from the Network tab in DevTools.
# The body contains empty filters and ordering, meaning we want all data with no filters applied.
headers_comuni = {
    'Content-Type': 'application/json-patch+json',
    # NOTE: the cookie below expires periodically.
# To update it, open https://situas.istat.it/web/#/territorio/body?id=74&dateFrom=2020-12-31
# click "Esporta", and copy the new Cookie value from the Network tab in DevTools (F12).
    'Cookie': 'rxVisitor=17823024965236V98Q9RG38MIGGGFKTR5GDCJ7UDA761E; dtSa=-; dtCookie=v_4_srv_8_sn_BOJED851GTSUKI3R96OSTBOQT022P6RT_perc_100000_ol_0_mul_1_app-3A5f9422794d4b081c_1; rxvt=1782307249186|1782304834351; dtPC=8$302496518_44h23vPPKDIRPBCQAKKWACIUBKRQRNDDPIOFKU-0e0'
}

body = {
    "orderFields": [],
    "orderDirects": [],
    "pFilterFields": [],
    "pFilterValues": []
}

# loop through the years from 2001 to 2024 and fetch the data for each year, if the file does not exist, save it to a CSV file
for anno in range(2001, 2025):
    if os.path.exists(f'../raw/comuni_{anno}.csv'):
        df_comuni = pd.read_csv(f'../raw/comuni_{anno}.csv', sep=';')
    else:
        url_comuni = f"https://situas.istat.it/ShibO2Module/api/Report/Spool/{anno}-12-31/74?&pdoctype=CSV"
        response_comuni = requests.post(url_comuni, headers=headers_comuni, json=body)
        if response_comuni.status_code == 200:
            data = response_comuni.content.decode('utf-8')
            df_comuni = pd.read_csv(io.StringIO(data), sep=';')
            df_comuni.to_csv(f'../raw/comuni_{anno}.csv', sep=';', index=False)
        else:
            print(f"Failed to fetch data for {anno}. Status code: {response_comuni.status_code}")


# merge the dataframes for each year into a single dataframe
df_comuni_list = []
for anno in range(2001, 2025):
    df_comuni = pd.read_csv(f'../raw/comuni_{anno}.csv', sep=';')
    df_comuni['Anno'] = anno
    df_comuni_list.append(df_comuni)

df_comuni= pd.concat(df_comuni_list, ignore_index=True)
df_comuni.head()

,Codice Ripartizione geografica,Codice Regione,Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente),Anno,Codice Provincia (Storico)
0,1,1,1,1001,1001,Agliè,NaN,TO,0,0,2574.0,2001,"13,1462",2001,2557,2001,2001,NaN
1,1,1,1,1002,1002,Airasca,NaN,TO,0,0,3554.0,2001,"15,7393",2001,3543,2001,2001,NaN
2,1,1,1,1003,1003,Ala di Stura,NaN,TO,0,0,479.0,2001,"46,3315",2001,480,2001,2001,NaN
3,1,1,1,1004,1004,Albiano d'Ivrea,NaN,TO,0,0,1696.0,2001,"11,7315",2001,1687,2001,2001,NaN
4,1,1,1,1005,1005,Alice Superiore,NaN,TO,0,0,616.0,2001,"7,3796",2001,619,2001,2001,NaN
